# Data Experiments — Finding the Optimal Trading Window

**Goal:** Using the best model from notebook 3 (LogisticRegression), experiment with data filtering to find the realistic trading window where the model has the most edge.

**Problem:** The 76.28% accuracy from notebook 3 is inflated because it includes snapshots from late in the candle (elapsed > 90%) where:
- The market has already priced in the outcome (UP bid > 0.90)
- No real trading opportunity exists (you'd pay $0.90 to win $1.00 — terrible R/R)
- The model is just reading the market's conclusion, not predicting anything

**Hypotheses to test:**
1. **Minimum elapsed time** — does the model need to see at least 1 min (20%) or 2 min (40%) of data to make useful predictions?
2. **Maximum market certainty** — if we filter out snapshots where the best bid > 0.85 (market already decided), how does accuracy change?
3. **Entry window** — is there a sweet spot (e.g., 30%-60% elapsed) where the model has enough signal but the market hasn't fully priced the outcome yet?
4. **Combined filters** — minimum elapsed + maximum bid = the realistic trading window

**Why this matters for the bot:**
A model that's 76% accurate on all snapshots but only 52% accurate during the actual entry window is useless. We need to know the accuracy *at the moment the bot would place a bet* — which is early enough for good R/R but late enough for signal.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from evaluator import Evaluator
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../data/latest_features.jsonl")

## 1. Load and prepare data

**What:** Load the feature dataset and prepare it the same way as notebook 3 — encode target, fill nulls, identify feature columns.

**Why:** We keep preprocessing identical to notebook 3 so results are directly comparable. The only thing that changes across experiments is *which rows* we include.

**Key columns for filtering:**
- `elapsed_pct` — how far into the candle (0.0 = start, 1.0 = end). For a 5-min candle: 0.2 = 1 min, 0.4 = 2 min, 0.6 = 3 min
- `up_best_bid` — market's implied probability for UP. > 0.85 means market is very confident UP wins
- `down_best_bid` — same for DOWN. > 0.85 means market is very confident DOWN wins
- The "winner bid" = max(up_best_bid, down_best_bid) — how certain the market is regardless of direction

In [ ]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)
df["winner_bid"] = df[["up_best_bid", "down_best_bid"]].max(axis=1)

# 31 optimal features (from notebook 1 Logistic forward selection)
feature_cols = sorted(
    [
        "bollinger_pct_b",
        "btc_direction_consistency",
        "btc_move_from_open",
        "btc_token_correlation",
        "btc_velocity",
        "candle_momentum",
        "consecutive_streak",
        "cross_book_flow",
        "current_elapsed",
        "depth_absorption_rate",
        "intra_candle_kurtosis",
        "liquidity_decay",
        "ob_pressure_gradient",
        "price_path_entropy",
        "prior_return",
        "prior_reversal_rate",
        "relative_volume",
        "reversal_regime",
        "rr_spread",
        "rsi",
        "smart_money_signal",
        "stochastic_k",
        "time_of_day_cos",
        "time_of_day_sin",
        "token_price_divergence",
        "trend_consistency",
        "up_book_imbalance",
        "up_spread_level",
        "volume_momentum",
        "volume_price_correlation",
        "weighted_mid_price",
    ]
)
df[feature_cols] = df[feature_cols].fillna(0.0)

print(f"Loaded {len(df):,} rows, {df['candle_id'].nunique()} candles, {len(feature_cols)} features")
print(f"Elapsed range: {df['elapsed_pct'].min():.2f} — {df['elapsed_pct'].max():.2f}")
print(f"Winner bid range: {df['winner_bid'].min():.2f} — {df['winner_bid'].max():.2f}")
print(f"UP rate: {df['target'].mean() * 100:.1f}%")

## 2. Understand the filtering landscape

**What:** Visualize how `elapsed_pct` and `winner_bid` are distributed and how they relate. This tells us how much data each filter removes.

**How to read the charts:**
- **Elapsed histogram:** Shows when during the candle we have data. If most snapshots are late (> 0.7), filtering to early snapshots will dramatically reduce our dataset.
- **Winner bid histogram:** Shows how confident the market is. A peak near 0.99 means many snapshots are "already decided" — these are the ones we want to filter out.
- **Scatter plot:** Each dot is a snapshot. X = elapsed %, Y = winner bid. The key insight is the relationship: as elapsed increases, winner bid should approach 1.0 (market converges to certainty). The "trading window" is the region with moderate elapsed AND moderate winner bid.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elapsed distribution
axes[0].hist(df["elapsed_pct"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(0.2, color="orange", linestyle="--", label="1 min (20%)")
axes[0].axvline(0.4, color="red", linestyle="--", label="2 min (40%)")
axes[0].set_title("Elapsed % Distribution")
axes[0].set_xlabel("Elapsed %")
axes[0].legend(fontsize=8)

# Winner bid distribution
axes[1].hist(df["winner_bid"].dropna(), bins=50, color="mediumpurple", edgecolor="white", alpha=0.8)
axes[1].axvline(0.85, color="red", linestyle="--", label="0.85 threshold")
axes[1].set_title("Winner Bid Distribution")
axes[1].set_xlabel("max(up_bid, down_bid)")
axes[1].legend(fontsize=8)

# Scatter: elapsed vs winner bid
sample_idx = np.random.choice(len(df), size=min(5000, len(df)), replace=False)
colors = ["#2ecc71" if o == 1 else "#e74c3c" for o in df["target"].iloc[sample_idx]]
axes[2].scatter(df["elapsed_pct"].iloc[sample_idx], df["winner_bid"].iloc[sample_idx], c=colors, alpha=0.2, s=3)
axes[2].axhline(0.85, color="red", linestyle="--", alpha=0.5)
axes[2].axvline(0.4, color="orange", linestyle="--", alpha=0.5)
axes[2].set_xlabel("Elapsed %")
axes[2].set_ylabel("Winner Bid")
axes[2].set_title("Elapsed vs Market Certainty (green=UP, red=DOWN)")

plt.tight_layout()
plt.show()

# Data retention stats
print("Data retention under different filters:")
for thresh in [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    n = (df["winner_bid"] <= thresh).sum()
    print(f"  winner_bid <= {thresh}: {n:>6,} rows ({n / len(df) * 100:.1f}%)")
print()
for pct in [0.2, 0.3, 0.4, 0.5, 0.6]:
    n = (df["elapsed_pct"] >= pct).sum()
    print(f"  elapsed >= {pct:.0%}:     {n:>6,} rows ({n / len(df) * 100:.1f}%)")

## 3. Helper: train and evaluate on filtered data

**What:** A reusable function that takes a data filter (boolean mask), trains LogisticRegression on the filtered data, and returns metrics.

**Why:** We'll run dozens of experiments with different filters. This function standardizes the process: same scaler, same split ratio, same random seed, same model — only the data mask changes.

**Important detail:** The scaler is fit on the *filtered* training data, not the full dataset. This is correct — in production, the model only sees data within the trading window.

In [ ]:
def run_experiment(mask, title, feature_cols=feature_cols, verbose=True):
    """Train LogisticRegression on filtered data, return metrics dict."""
    df_f = df[mask].copy()
    if len(df_f) < 100:
        return {"title": title, "n": len(df_f), "accuracy": None, "mse": None, "r2": None, "f1": None}

    X_f = df_f[feature_cols].values
    y_f = df_f["target"].values

    # Check class balance
    up_rate = y_f.mean()
    if up_rate < 0.05 or up_rate > 0.95:
        if verbose:
            print(f"  {title}: skipped — class imbalance ({up_rate * 100:.1f}% UP)")
        return {"title": title, "n": len(df_f), "accuracy": None, "mse": None, "r2": None, "f1": None}

    n_candles = df_f["candle_id"].nunique()
    if n_candles < 10:
        if verbose:
            print(f"  {title}: skipped — only {n_candles} candles")
        return {"title": title, "n": len(df_f), "accuracy": None, "mse": None, "r2": None, "f1": None}

    # Time-based split (80/20)
    candle_ids = df_f["candle_id"].unique()
    split_idx = int(len(candle_ids) * 0.8)
    train_ids = set(candle_ids[:split_idx])
    train_mask = df_f["candle_id"].isin(train_ids)

    scaler_f = StandardScaler()
    X_f = scaler_f.fit_transform(X_f)

    X_tr, X_te = X_f[train_mask], X_f[~train_mask]
    y_tr, y_te = y_f[train_mask], y_f[~train_mask]

    if len(y_te) < 10:
        return {"title": title, "n": len(df_f), "accuracy": None, "mse": None, "r2": None, "f1": None}

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)

    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = model.predict(X_te)

    ev = Evaluator(y_te, y_pred, y_prob, title=title)

    if verbose:
        print(
            f"  {title}: n={len(df_f):,} acc={ev.accuracy * 100:.2f}% MSE={ev.mse:.4f} R²={ev.r2 * 100:.1f}% F1={ev.f1 * 100:.1f}%"
        )

    return {
        "title": title,
        "n": len(df_f),
        "n_test": len(y_te),
        "accuracy": ev.accuracy,
        "mse": ev.mse,
        "r2": ev.r2,
        "f1": ev.f1,
        "up_rate": up_rate,
        "evaluator": ev,
    }

## 4. Experiment 1 — Minimum elapsed time

**Hypothesis:** The model needs to see enough intra-candle data to compute meaningful indicators (velocity, imbalance trends, etc.). With only 1-2 snapshots, these indicators return None (filled as 0). But too late and the market has already decided.

**Test:** Train separate models filtering to snapshots where `elapsed_pct >= X` for X in [0%, 10%, 20%, ..., 80%].

**What we expect:**
- Accuracy should **increase** as we raise the minimum — more data = better indicators
- At some point accuracy will plateau or even increase sharply as we approach the candle close (market certainty)
- The interesting region is where accuracy is high but we're still early enough to trade

In [ ]:
print("Experiment 1: Minimum elapsed time")
print("=" * 70)

min_elapsed_results = []
for min_pct in np.arange(0.0, 0.85, 0.05):
    mask = df["elapsed_pct"] >= min_pct
    result = run_experiment(mask, f"elapsed >= {min_pct:.0%}")
    result["min_elapsed"] = min_pct
    min_elapsed_results.append(result)

In [ ]:
valid = [r for r in min_elapsed_results if r["accuracy"] is not None]
x = [r["min_elapsed"] * 100 for r in valid]
accs = [r["accuracy"] * 100 for r in valid]
ns = [r["n"] for r in valid]

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(x, accs, "o-", color="steelblue", linewidth=2, markersize=8, label="Accuracy")
ax1.set_xlabel("Minimum Elapsed %")
ax1.set_ylabel("Accuracy (%)", color="steelblue")
ax1.axhline(50, color="red", linestyle="--", alpha=0.3)

ax2 = ax1.twinx()
ax2.bar(x, ns, width=3, alpha=0.15, color="gray", label="Sample count")
ax2.set_ylabel("Rows available", color="gray")

ax1.set_title("Accuracy vs Minimum Elapsed Time")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

## 5. Experiment 2 — Maximum market certainty (winner bid cap)

**Hypothesis:** Snapshots where the market is already very confident (winner bid > 0.85) are "free" predictions — the model gets them right but there's no trading edge because:
1. The entry cost is too high (paying $0.90 to win $1.00 = only 11% return)
2. The model isn't really predicting — it's just reading the market's conclusion

**Test:** Cap the maximum winner bid at [0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0] and retrain.

**What we expect:**
- Accuracy should **drop** as we lower the cap — we're removing the "easy" predictions
- The question is: *how much does it drop?* If accuracy goes from 76% to 55%, the model only worked on easy cases. If it stays at 65%, there's real signal even in uncertain markets.
- R² is especially important here — it measures how well the model estimates *probabilities*, not just binary accuracy

In [ ]:
print("Experiment 2: Maximum winner bid")
print("=" * 70)

max_bid_results = []
for max_bid in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]:
    mask = df["winner_bid"] <= max_bid
    result = run_experiment(mask, f"winner_bid <= {max_bid:.2f}")
    result["max_bid"] = max_bid
    max_bid_results.append(result)

In [ ]:
valid = [r for r in max_bid_results if r["accuracy"] is not None]
x = [r["max_bid"] for r in valid]
accs = [r["accuracy"] * 100 for r in valid]
r2s = [r["r2"] * 100 for r in valid]
ns = [r["n"] for r in valid]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(x, accs, "o-", color="steelblue", linewidth=2, markersize=8)
ax1.set_xlabel("Maximum Winner Bid")
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy vs Max Winner Bid")
ax1.axhline(50, color="red", linestyle="--", alpha=0.3, label="random")
ax1.axvline(0.85, color="orange", linestyle="--", alpha=0.5, label="0.85 threshold")
ax1.legend()

ax2 = axes[1]
ax2.plot(x, r2s, "s-", color="#2ecc71", linewidth=2, markersize=8)
ax2.set_xlabel("Maximum Winner Bid")
ax2.set_ylabel("R² (%)")
ax2.set_title("R² vs Max Winner Bid")
ax2.axhline(0, color="red", linestyle="--", alpha=0.3)
ax2.axvline(0.85, color="orange", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## 6. Experiment 3 — Entry windows (elapsed range)

**Hypothesis:** There's an optimal *window* within the candle — early enough that the market hasn't converged, but late enough that we have enough snapshots for good indicators.

**Test:** Sliding windows of 20% width: [0-20%, 10-30%, 20-40%, ..., 70-90%]. Each window represents a different entry timing strategy.

**What we expect:**
- Very early windows (0-20%) should have low accuracy — not enough data, indicators are mostly None
- Very late windows (80-100%) should have high accuracy — market already decided
- The sweet spot should be somewhere in the middle — enough signal, market still uncertain
- Window size matters too: 10%, 20%, 30% widths may give different results

In [ ]:
print("Experiment 3: Entry windows (20% width)")
print("=" * 70)

window_results = []
for start in np.arange(0.0, 0.85, 0.05):
    end = start + 0.20
    mask = (df["elapsed_pct"] >= start) & (df["elapsed_pct"] < end)
    result = run_experiment(mask, f"[{start:.0%}-{end:.0%})")
    result["start"] = start
    result["end"] = end
    window_results.append(result)

In [ ]:
valid = [r for r in window_results if r["accuracy"] is not None]
labels = [r["title"] for r in valid]
accs = [r["accuracy"] * 100 for r in valid]
ns = [r["n"] for r in valid]

fig, ax1 = plt.subplots(figsize=(14, 6))
bars = ax1.bar(range(len(labels)), accs, color="steelblue", edgecolor="white", alpha=0.8)
ax1.set_xticks(range(len(labels)))
ax1.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Accuracy by Entry Window (20% width)")
ax1.axhline(50, color="red", linestyle="--", alpha=0.3)

# Label sample counts
for bar, n in zip(bars, ns, strict=True):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"n={n:,}",
        ha="center",
        va="bottom",
        fontsize=7,
        rotation=45,
    )

plt.tight_layout()
plt.show()

## 7. Experiment 4 — Combined filter (realistic trading scenario)

**Hypothesis:** The real trading window combines both constraints:
1. **Enough elapsed time** to compute meaningful indicators (at least 1-2 min)
2. **Market not yet decided** (winner bid <= 0.85) so there's still a profitable entry

**Test:** Grid search over min_elapsed and max_winner_bid combinations.

**What we expect:** A heatmap showing the accuracy landscape. The top-right corner (high elapsed, high bid cap) is unrealistic trading. The bottom-left (low elapsed, low bid cap) has too little data. The sweet spot is somewhere in between.

**How to read the heatmap:**
- **X-axis** = minimum elapsed %
- **Y-axis** = maximum winner bid
- **Color** = accuracy (darker = better)
- Look for the region with the best accuracy that still has enough data and a realistic entry point

In [ ]:
print("Experiment 4: Combined filter grid search")
print("=" * 70)

elapsed_vals = np.arange(0.0, 0.75, 0.10)
bid_vals = [0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]

grid_results = []
for min_e in tqdm(elapsed_vals, desc="Grid search"):
    for max_b in bid_vals:
        mask = (df["elapsed_pct"] >= min_e) & (df["winner_bid"] <= max_b)
        result = run_experiment(mask, f"e>={min_e:.0%} b<={max_b:.2f}", verbose=False)
        result["min_elapsed"] = min_e
        result["max_bid"] = max_b
        grid_results.append(result)

In [ ]:
# Build heatmap matrices
acc_matrix = np.full((len(bid_vals), len(elapsed_vals)), np.nan)
n_matrix = np.full((len(bid_vals), len(elapsed_vals)), 0)

for r in grid_results:
    i = bid_vals.index(r["max_bid"])
    j = list(elapsed_vals).index(r["min_elapsed"])
    if r["accuracy"] is not None:
        acc_matrix[i, j] = r["accuracy"] * 100
    n_matrix[i, j] = r["n"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy heatmap
im1 = axes[0].imshow(acc_matrix, cmap="RdYlGn", aspect="auto", vmin=50, vmax=max(acc_matrix[~np.isnan(acc_matrix)]))
axes[0].set_xticks(range(len(elapsed_vals)))
axes[0].set_xticklabels([f"{v:.0%}" for v in elapsed_vals])
axes[0].set_yticks(range(len(bid_vals)))
axes[0].set_yticklabels([f"{v:.2f}" for v in bid_vals])
axes[0].set_xlabel("Minimum Elapsed %")
axes[0].set_ylabel("Maximum Winner Bid")
axes[0].set_title("Accuracy (%)")
for i in range(len(bid_vals)):
    for j in range(len(elapsed_vals)):
        val = acc_matrix[i, j]
        if not np.isnan(val):
            axes[0].text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7)
plt.colorbar(im1, ax=axes[0])

# Sample count heatmap
im2 = axes[1].imshow(n_matrix, cmap="Blues", aspect="auto")
axes[1].set_xticks(range(len(elapsed_vals)))
axes[1].set_xticklabels([f"{v:.0%}" for v in elapsed_vals])
axes[1].set_yticks(range(len(bid_vals)))
axes[1].set_yticklabels([f"{v:.2f}" for v in bid_vals])
axes[1].set_xlabel("Minimum Elapsed %")
axes[1].set_ylabel("Maximum Winner Bid")
axes[1].set_title("Sample Count")
for i in range(len(bid_vals)):
    for j in range(len(elapsed_vals)):
        axes[1].text(j, i, f"{n_matrix[i, j]:,}", ha="center", va="center", fontsize=6)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

## 8. Best realistic configuration

**What:** Find the best filter combination that has at least 1,000 test samples (for statistical reliability) and a maximum winner bid <= 0.85 (realistic entry cost).

**Why:** A filter with 95% accuracy on 50 samples is meaningless — the confidence interval is huge. We need enough data for the accuracy number to be trustworthy, AND the filter must represent a realistic trading scenario.

In [ ]:
# Find best realistic configuration
realistic = [
    r
    for r in grid_results
    if r["accuracy"] is not None and r["max_bid"] <= 0.85 and r["n"] >= 5000  # enough data
]

if realistic:
    best = max(realistic, key=lambda r: r["accuracy"])
    print("BEST REALISTIC CONFIGURATION")
    print("=" * 60)
    print(f"  Filter:   elapsed >= {best['min_elapsed']:.0%}, winner_bid <= {best['max_bid']:.2f}")
    print(f"  Samples:  {best['n']:,} total, {best['n_test']:,} test")
    print(f"  Accuracy: {best['accuracy'] * 100:.2f}%")
    print(f"  MSE:      {best['mse']:.4f}")
    print(f"  R²:       {best['r2'] * 100:.1f}%")
    print(f"  F1:       {best['f1'] * 100:.1f}%")
    print(f"  UP rate:  {best['up_rate'] * 100:.1f}%")

    # Show full report with charts
    best["evaluator"].full_report()
else:
    print("No configuration met the criteria (>= 5000 samples, winner_bid <= 0.85)")

## 9. Experiment 5 — R/R adjusted accuracy

**What:** Accuracy alone doesn't capture trading profitability. A correct prediction at entry cost $0.50 (R/R = 1:1) is worth more than one at $0.80 (R/R = 0.25:1). Here we simulate the actual PnL.

**How it works:**
For each snapshot in the test set:
- If model predicts UP and outcome is UP → profit = 1.0 - entry_cost
- If model predicts UP and outcome is DOWN → loss = -entry_cost
- If model predicts DOWN (no bet) → PnL = 0
- Entry cost = UP token ask price (what you'd actually pay)

**Why this matters:** A model with 60% accuracy at average entry $0.55 is more profitable than one with 70% accuracy at average entry $0.85. This experiment reveals whether the model's edge translates to real profit.

In [ ]:
print("Experiment 5: Simulated PnL by entry window")
print("=" * 70)

pnl_results = []
for max_bid in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    mask = (df["winner_bid"] <= max_bid) & (df["elapsed_pct"] >= 0.20)
    df_f = df[mask].copy()
    if len(df_f) < 500:
        continue

    X_f = StandardScaler().fit_transform(df_f[feature_cols].values)
    y_f = df_f["target"].values
    asks = df_f["up_best_ask"].values

    # Time-based split
    cids = df_f["candle_id"].unique()
    split = int(len(cids) * 0.8)
    train_mask = df_f["candle_id"].isin(set(cids[:split])).values
    X_tr, X_te = X_f[train_mask], X_f[~train_mask]
    y_tr, y_te = y_f[train_mask], y_f[~train_mask]
    asks_te = asks[~train_mask]

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)

    # Simulate PnL: only bet when model says UP
    total_pnl = 0.0
    n_bets = 0
    wins = 0
    for pred, truth, ask in zip(preds, y_te, asks_te, strict=True):
        if pred == 1 and np.isfinite(ask) and ask > 0:
            n_bets += 1
            if truth == 1:
                total_pnl += 1.0 - ask
                wins += 1
            else:
                total_pnl -= ask

    avg_ask = np.mean([a for p, a in zip(preds, asks_te, strict=True) if p == 1 and a is not None])
    win_rate = wins / n_bets if n_bets > 0 else 0
    roi = total_pnl / (n_bets * avg_ask) * 100 if n_bets > 0 and avg_ask > 0 else 0

    print(
        f"  bid<={max_bid:.2f}: {n_bets:>5} bets, win_rate={win_rate * 100:.1f}%, "
        f"avg_entry=${avg_ask:.3f}, PnL=${total_pnl:+.2f}, ROI={roi:+.1f}%"
    )

    pnl_results.append(
        {
            "max_bid": max_bid,
            "n_bets": n_bets,
            "win_rate": win_rate,
            "avg_entry": avg_ask,
            "total_pnl": total_pnl,
            "roi": roi,
        }
    )

In [ ]:
if pnl_results:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    x = [r["max_bid"] for r in pnl_results]

    # Win rate
    axes[0].plot(x, [r["win_rate"] * 100 for r in pnl_results], "o-", color="steelblue", linewidth=2)
    axes[0].axhline(50, color="red", linestyle="--", alpha=0.3)
    axes[0].set_xlabel("Max Winner Bid")
    axes[0].set_ylabel("Win Rate (%)")
    axes[0].set_title("Win Rate by Entry Window")

    # Total PnL
    colors = ["#2ecc71" if r["total_pnl"] > 0 else "#e74c3c" for r in pnl_results]
    axes[1].bar(x, [r["total_pnl"] for r in pnl_results], width=0.04, color=colors, edgecolor="white")
    axes[1].axhline(0, color="gray", linestyle="-", linewidth=0.5)
    axes[1].set_xlabel("Max Winner Bid")
    axes[1].set_ylabel("Total PnL ($)")
    axes[1].set_title("Simulated PnL")

    # ROI
    colors = ["#2ecc71" if r["roi"] > 0 else "#e74c3c" for r in pnl_results]
    axes[2].bar(x, [r["roi"] for r in pnl_results], width=0.04, color=colors, edgecolor="white")
    axes[2].axhline(0, color="gray", linestyle="-", linewidth=0.5)
    axes[2].set_xlabel("Max Winner Bid")
    axes[2].set_ylabel("ROI (%)")
    axes[2].set_title("Return on Investment")

    plt.tight_layout()
    plt.show()

---

## 10. Conclusion

### What the Experiments Show

The experiments confirm the same core pattern as before: accuracy improves as elapsed % increases, and filtering on `winner_bid` removes "easy" late-candle predictions that inflate headline numbers. See the results above for specific accuracy and ROI figures from the current run.

### Key Findings (consistent across runs)

**1. Elapsed % is the strongest predictor of accuracy.**
Accuracy rises monotonically from the start of the candle to the end — each additional 20% elapsed adds signal. Early entries (< 40% elapsed) are harder but offer better risk/reward if entry prices are still cheap.

**2. `winner_bid` filtering reveals true signal.**
Without the bid cap, accuracy appears high because the model is reading a market that has already converged. With `winner_bid <= 0.85`, the realistic edge drops to the 60–70% range — still meaningful, but much more honest.

**3. Best ROI comes from selectivity, not accuracy.**
Lower bid caps (0.65–0.70) reduce the number of bets but improve entry prices enough to boost ROI above higher-cap configurations. See the PnL simulation results above for current numbers.

**4. Recommended trading configuration (reference outputs above for current thresholds):**

| Parameter | Guidance |
|-----------|----------|
| Minimum elapsed | ≥ 20–40% (see Exp 1 results) |
| Maximum winner bid | 0.65–0.85 (see Exp 2/5 results) |
| Model | LogisticRegression |
| Expected win rate | See results above |
| Expected ROI | See PnL simulation above |

### Next Steps

1. **Temporal validation** — walk-forward split to confirm results hold on future data
2. **Richer features** — 10-level orderbook data from the new `DataCollector` may improve edge in uncertain markets
3. **Dynamic entry** — enter when confidence threshold is crossed rather than at a fixed elapsed %


In [ ]:
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)

print("\n1. MINIMUM ELAPSED TIME:")
for r in min_elapsed_results:
    if r["accuracy"] is not None:
        print(f"   {r['title']:<20} acc={r['accuracy'] * 100:.1f}%  n={r['n']:>6,}")

print("\n2. MAXIMUM WINNER BID:")
for r in max_bid_results:
    if r["accuracy"] is not None:
        print(f"   {r['title']:<25} acc={r['accuracy'] * 100:.1f}%  R²={r['r2'] * 100:.1f}%  n={r['n']:>6,}")

print("\n3. ENTRY WINDOWS (20% width):")
for r in window_results:
    if r["accuracy"] is not None:
        print(f"   {r['title']:<15} acc={r['accuracy'] * 100:.1f}%  n={r['n']:>6,}")

if realistic:
    best = max(realistic, key=lambda r: r["accuracy"])
    print("\n4. BEST REALISTIC CONFIG:")
    print(f"   elapsed >= {best['min_elapsed']:.0%}, winner_bid <= {best['max_bid']:.2f}")
    print(f"   accuracy={best['accuracy'] * 100:.2f}%  R²={best['r2'] * 100:.1f}%  n={best['n']:,}")

if pnl_results:
    best_pnl = max(pnl_results, key=lambda r: r["roi"])
    print("\n5. BEST PnL CONFIG:")
    print(f"   winner_bid <= {best_pnl['max_bid']:.2f}, elapsed >= 20%")
    print(
        f"   win_rate={best_pnl['win_rate'] * 100:.1f}%  avg_entry=${best_pnl['avg_entry']:.3f}  "
        f"ROI={best_pnl['roi']:+.1f}%  PnL=${best_pnl['total_pnl']:+.2f}"
    )

print("\n" + "=" * 70)
print("KEY TAKEAWAYS:")
print("  - Unfiltered accuracy (76%) is misleading — includes late-candle certainty")
print("  - The realistic trading window is where accuracy AND R/R are both viable")
print("  - PnL simulation reveals whether model edge survives after entry costs")
print("  - Next: collect more data in the realistic window with new DataCollector")